<a href="https://colab.research.google.com/github/lapshinaaa/recsys-tasks/blob/main/DeepRecSys4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep RecSys Course
## Practice notebook $4$

In this assignment, you will implement a candidate retrieval recommender model based on the Transformer architecture. To do this, you will need to write a dataset, a Transformer-based architecture, the model itself that predicts the next item from the user's history, as well as the train and evaluation loops.


### Data

The data is stored in the `data.zip` archive, which consists of:
* `interactions.parquet` - user-item interactions from the Yambda dataset (likes for the 500m version)
* `embeddings.parquet` - already filtered and slightly more densely packed track embeddings from Yambda
* `artists.parquet` - item metadata with mapping to artists

In this assignment, we will only be interested in the interactions file, `interactions.parquet`.

You can download the archive here: [link to Google Drive](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). In the next block, we download the dataset automatically, so you do not need to download it manually.


## Downloading the data

In [1]:
!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -oq dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=dd78d93b-22c7-46b6-be40-d859fb1fb27e
To: /content/dataset.zip
100% 356M/356M [00:07<00:00, 46.2MB/s]


## Import libraries

In [2]:
from collections import defaultdict
import gc
import os
from typing import Callable, Dict, List, Tuple, Any, Optional

import numpy as np
import polars as pl

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import tests

In [3]:
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
DEVICE = "cuda"

# 0. Prepping data and metrics

Data processing

In [4]:
# paths to data
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# global vars
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# for reproducibility
np.random.seed(42)

data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################


Metrics

In [5]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################

    """
    Per-user metrics. targets = relevant items G_u, candidates = ranked list R_u (length >= topk).
    Returns hitrate@k(u), recall@k(u), ndcg@k(u).
    """

    recs = candidates[:topk]
    gt = set(targets)

    hits = [1 if item in gt else 0 for item in recs] # faster since gt is a dict
    num_hits = sum(hits)

    # Hitrate@K(u)
    hitrate = 1.0 if num_hits > 0 else 0.0

    # Recall@K(u)
    denom = min(len(targets), topk) # for len can't use len, MUST use original targets
    recall = (num_hits / denom) if denom > 0 else 0.0

    # DCG@K(u)
    dcg = 0.0
    for idx, h in enumerate(hits, start=1):  # idx = 1..K
        if h:
            dcg += 1.0 / math.log2(idx + 1)

    # iDCG@K(u)
    idcg = 0.0
    for idx in range(1, denom + 1):
        idcg += 1.0 / math.log2(idx + 1)

    ndcg = (dcg / idcg) if idcg > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg}


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    """
    Aggregates metrics across users and computes coverage@K.
    Assumes candidates[uid] has length at least topk (or exactly topk, as your note says).
    """
    uids = list(targets.keys())

    hitrate_sum = 0.0
    recall_sum = 0.0
    ndcg_sum = 0.0

    # coverage: union of all recommended items across users in topK
    covered_items = set()

    for uid in uids:
        gt_u = targets[uid]
        rec_u = candidates[uid][:topk]  # guarantee topk slice

        m = get_metrics(gt_u, rec_u, topk=topk)
        hitrate_sum += m["hitrate"]
        recall_sum += m["recall"]
        ndcg_sum += m["ndcg"]

        covered_items.update(rec_u)

    num_users = len(uids)
    hitrate = hitrate_sum / num_users if num_users > 0 else 0.0
    recall = recall_sum / num_users if num_users > 0 else 0.0
    ndcg = ndcg_sum / num_users if num_users > 0 else 0.0

    coverage = (len(covered_items) / catalog_size) if catalog_size > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg, "coverage": coverage}

# 1. Implement datasets for training and validation and collator for them

Similarly to the second homework assignment, in this part you need to implement helper functions for working with user histories of variable length. These functions will be used when constructing the dataset, forming batches, and feeding data into the model, so it is important to define the sequence representation format in advance. Some of the solutions from Homework 2 can be reused here.

#### What conceptually changes compared with Homework 2

In the second homework assignment, we created many training examples from a single user. If the user had `n` different interactions, then there were `n - 1` training examples in total. Each example had the form `<target event, history before this event>`. If we train a computationally expensive user encoder, such as a Transformer, this approach would require us to run it `n` times for nearly the same user history.

User history: `[e1, e2, e3, e4, e5]`.

Samples:
1. `<target: e2, history: [e1]>`
2. `<target: e3, history: [e1, e2]>`
3. `<target: e4, history: [e1, e2, e3]>`
4. `<target: e5, history: [e1, e2, e3, e4]>`

However, if we use a causal Transformer as the encoder — it is also often called a decoder — and use the hidden state of the latest event as the user's vector representation, or apply some simple aggregation over the hidden states in the history, then we can train much more efficiently in the style of large language models.

More specifically, we can apply the Transformer once to the event sequence `[e1, e2, e3, e4]` and obtain user hidden states for all samples at once:
- `history: [e1]`
- `history: [e1, e2]`
- `history: [e1, e2, e3]`
- and so on.

Why does this work? Because the Transformer is causal: when encoding the current event, or the history available at that event, it can only attend to past events.

If we can obtain user vectors for all of a user's samples in a single forward pass, then we can also train on all of them at once by computing the loss for every sample belonging to that user and averaging it.

#### Reminder: Flattened History Representation

When several user histories are combined into a batch, they are represented in flattened format: all sequences are concatenated into one shared tensor, while their boundaries are specified by a separate `length` tensor.

Suppose several users are included in one batch. Their histories are combined into one flat tensor of the following form:

`[u1_t1, u1_t2, ..., u1_tL1, u2_t1, ..., u2_tL2, ...]`

The interactions of user 1 are written first, followed by the interactions of user 2, and so on.

To preserve the boundaries between users, a separate `length` tensor stores the number of history elements belonging to each user in the batch:

`length = [L1, L2, ..., LB]`

where `B` is the batch size and `Li` is the length of the history of the `i`-th user.

Therefore:
- at the dataset level, one object corresponds to the complete history of one user;
- at the batch level, several such histories are combined into flattened format;
- the `length` tensor makes it possible to reconstruct the boundaries of the individual sequences.

#### What Needs to Be Implemented

Your task is to implement functions that:
- convert the flattened representation into padded format together with a mask;
- implement the logic for creating the training and validation datasets;
- prepare a batch to be fed into the model.

### Function `create_masked_tensor` (from DeepRecSys2)

In [6]:
def get_mask(lengths: torch.Tensor) -> torch.Tensor:
  """
    Creates a boolean mask for variable-length sequences.
  """

 # batch_size = lengths.size(0)
  max_len = lengths.max().item()

  positions = torch.arange(max_len, device=lengths.device)
  mask = positions.unsqueeze(0) < lengths.unsqueeze(1)

  return mask

In [7]:
def create_masked_tensor(data: torch.Tensor, lengths: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
  """
  Converts a batch of flattened variable-length sequences into a padded tensor and mask.
  Supports:
    - indices: data shape (total_num_elements,)
    - embeddings/features: data shape (total_num_elements, d1, d2, ...)

  Parameters
  ----------
  data : torch.Tensor
      Input tensor containing flattened sequences:
      - For indices: shape (total_num_elements,)
      - For embeddings: shape (total_num_elements, embedding_dim)
  lengths : torch.Tensor
      1D tensor of sequence lengths, shape (batch_size,). Specifies the actual length
      of each sequence.

  Returns
  -------
  Tuple[torch.Tensor, torch.Tensor]
      - padded_tensor: Padded tensor of shape:
          - (batch_size, max_seq_len) for indices
          - (batch_size, max_seq_len, embedding_dim) for embeddings
          Shorter sequences are right-padded with zeros.
      - mask: Boolean mask of shape (batch_size, max_seq_len) where True indicates
          valid elements and False indicates padding. Can be used in attention or loss computation.

  Examples
  --------
  >>> data = torch.tensor([1, 2, 3, 4, 5, 6])  # sequences: [1,2], [3,4,5], [6]
  >>> lengths = torch.tensor([2, 3, 1])
  >>> padded, mask = create_masked_tensor(data, lengths)
  >>> padded
  tensor([[1, 2, 0],
          [3, 4, 5],
          [6, 0, 0]])
  >>> mask
  tensor([[ True,  True, False],
          [ True,  True,  True],
          [ True, False, False]])
  """
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################

  mask = get_mask(lengths)
  batch = lengths.size(0)
  elem_max = int(lengths.max().item())

  if data.dim() == 1:
    padded = torch.zeros(batch, elem_max, dtype=data.dtype, device=data.device)
  else:
    D = data.shape[1:]
    padded = torch.zeros(batch, elem_max, *D, dtype=data.dtype, device=data.device)

  start = 0
  for i, length in enumerate(lengths.tolist()):
    end = start + length
    padded[i, :length] = data[start:end]
    start = end

  return padded, mask

In [8]:
tests.test_create_masked_tensor(create_masked_tensor)

All good! :)


### Dataset Classes

Implement the training and validation dataset classes that work with user interaction histories and prepare samples.

#### Main Idea

When working with Transformers, we define the maximum sequence length — the context size that we want the model to handle. This length is chosen according to different constraints: computational resources, because a very large context takes too long to process or may not fit into GPU memory, and model quality, because in some cases a model may perform worse with a very long context.

In the simplest case, all users have histories shorter than the specified limit. In practice, however, some users have longer histories that cannot be processed by the Transformer in a single forward pass.

In that case, we create chunks: the user's history is divided into segments that do not exceed the maximum length, and each segment is used as a training example. In this assignment, you are asked to divide the user history into non-overlapping segments.

For example, if the user's history is

`[e1, e2, e3, e4, e5]`

and the maximum sequence length is 2, the samples will be:

1. `history: [e1, e2], targets: [e2, e3]`
2. `history: [e2, e3], targets: [e3, e4]`
3. `history: [e3, e4], targets: [e4, e5]`

In this assignment, the training and validation logic is not combined within a single dataset class. Instead, you need to implement two separate datasets:

- `YambdaTrainDataset` — for training;
- `YambdaEvalDataset` — for validation.

This makes the behavior of the datasets more transparent: the `train` dataset is responsible only for preparing training examples, while the `eval` dataset is responsible only for preparing users for model evaluation.

#### `YambdaTrainDataset`

The training dataset is built only from `histories`.

For each user, the history is divided into chunks of length no greater than `max_seq_len`, moving from the beginning of the history toward the end.

Each sample must have the following format:

```python
{
  "uid": uid,
  "history": List[int],
  "targets": List[int],
  "length": int
}
```

where:

- `history` — the input sequence of items in chronological order;
- `targets` — the targets shifted by one step for `next-item prediction`;
- `length` — the length of the history and, correspondingly, the targets.

#### `YambdaEvalDataset`

The validation dataset is built from `histories` and `targets`.

There must be exactly one sample per user. Only users who are present in `targets` should be included in the dataset.

For each such user, take only the tail of their history, with length no greater than `max_seq_len`.

Unlike `YambdaTrainDataset`, here you do not need to divide the history into chunks and do not need to construct shifted targets.

Each sample must have the following format:

```python
{
  "uid": uid,
  "history": List[int],
  "length": int
}
```

where:

- `history` — the tail of the user's history in chronological order;
- `length` — the length of this history.

In [9]:
class YambdaTrainDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

        self.samples = []

        for uid, events in histories.items():
          if len(events) < 2:
            continue

          input_sequence = events[:-1]
          target_sequence = events[1:]

          for start in range(0, len(input_sequence), max_seq_len):
                end = start + max_seq_len

                history_chunk = input_sequence[start:end]
                target_chunk = target_sequence[start:end]

                if len(history_chunk) == 0:
                    continue

                self.samples.append(
                    {
                        "uid": uid,
                        "history": history_chunk,
                        "targets": target_chunk,
                        "length": len(history_chunk),
                    }
                )


    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [10]:
tests.test_yambda_train_dataset(YambdaTrainDataset)

All good! :)


In [11]:
class YambdaEvalDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        targets: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

        self.samples = []
        for uid in targets:
            if uid not in histories:
                continue

            history = histories[uid][-max_seq_len:]

            if len(history) == 0:
                continue

            self.samples.append(
                {
                    "uid": uid,
                    "history": history,
                    "length": len(history),
                }
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]


In [12]:
tests.test_yambda_eval_dataset(YambdaEvalDataset)

All good! :)


### Function `collate_fn`

Implement the `collate_fn` function, which will be used in the `DataLoader` to convert a list of samples from `YambdaTrainDataset` or `YambdaEvalDataset` into batches that can be fed into the model and used for further processing.

As mentioned earlier, we use a flattened representation:

- concatenate all user histories in the batch into a single 1D tensor;
- store `lengths` separately so that the sequence boundaries can be reconstructed later.

#### Input

`batch: List[Dict[str, Any]]` — a list of samples from `YambdaTrainDataset` or `YambdaEvalDataset`.

#### What `collate_fn` Should Do

The function should:

- create a single dictionary in which all values are `torch.Tensor` objects of type `torch.long`;
- not use padding; only flattened concatenation together with `lengths`;
- preserve the order of the objects in `batch` during concatenation;
- convert all numerical values to `torch.Tensor` objects of type `torch.long`.

In [13]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################

    result = {
        "uid": torch.tensor(
            [sample["uid"] for sample in batch],
            dtype=torch.long,
        ),
        "history": torch.tensor(
            [
                item
                for sample in batch
                for item in sample["history"]
            ],
            dtype=torch.long,
        ),
        "length": torch.tensor(
            [sample["length"] for sample in batch],
            dtype=torch.long,
        ),
    }

    # training samples also contain targets
    if "targets" in batch[0]:
        result["targets"] = torch.tensor(
            [
                item
                for sample in batch
                for item in sample["targets"]
            ],
            dtype=torch.long,
        )

    return result

In [14]:
tests.test_collate_fn(collate_fn)

All good! :)


```text
Individual dataset samples
        ↓
DataLoader collects several samples
        ↓
collate_fn
        ↓
Flattened batch:
- histories are concatenated into one 1D tensor
- targets are concatenated in the same order for training
- lengths preserve sequence boundaries
        ↓
create_masked_tensor
        ↓
Rectangular padded tensor + padding mask
        ↓
Causal Transformer
```

Let's take a look at the result

### Variables to Define

- `data` — the already preprocessed interactions dataframe containing the columns `uid`, `item_id`, and `timestamp`.
- `train_df` — the rows from `data` where `timestamp < test_start_ts`.
- `test_df` — the rows from `data` where `timestamp >= test_start_ts`.
- `catalog_size` — the number of unique `item_id` values in `data`.

Important: you do not need to remap `item_id` through `item_mapping` again if this has already been done earlier in the notebook.

In [15]:
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

data = (
    data
    .select(["uid", "item_id", "timestamp"])
)

popular_items = (
    data
    .group_by("item_id")
    .len()
    .filter(pl.col("len") >= CORE_MIN_INTERACTIONS_PER_ITEM)
    .select("item_id")
)

data = data.join(
    popular_items,
    on="item_id",
    how="semi",
)

item_mapping = (
    data
    .select("item_id")
    .unique()
    .sort("item_id")
    .with_row_index("mapped_item_id")
)

data = (
    data
    .join(item_mapping, on="item_id", how="left")
    .drop("item_id")
    .rename({"mapped_item_id": "item_id"})
    .sort(["uid", "timestamp"])
)

max_ts = data.select(pl.col("timestamp").max()).item()
test_start_ts = max_ts - TEST_INTERVAL_SECONDS

train_df = data.filter(
    pl.col("timestamp") < test_start_ts
)

test_df = data.filter(
    pl.col("timestamp") >= test_start_ts
)

catalog_size = data["item_id"].n_unique()

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

In [16]:
tests.test_split_fields(
    data=data,
    train_df=train_df,
    test_df=test_df,
    catalog_size=catalog_size,
)

All good! :)


In [17]:
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
CATALOG_SIZE = data["item_id"].n_unique()

histories = {
    row["uid"]: list(row["item_id"])
    for row in train_df.group_by("uid", maintain_order=True)
    .agg(pl.col("item_id").sort_by("timestamp"))
    .iter_rows(named=True)
}

raw_targets = dict(test_df.group_by("uid").agg(pl.col("item_id")).iter_rows())
targets = {
    uid: t
    for uid, t in raw_targets.items()
    if uid in histories and len(histories[uid]) > 0
}


yambda_train_dataset = YambdaTrainDataset(histories=histories)
yambda_eval_dataset = YambdaEvalDataset(histories=histories, targets=targets)

yambda_train_dataloader = DataLoader(
    dataset=yambda_train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True,
)

yambda_eval_dataloader = DataLoader(
    dataset=yambda_eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False,
)

In [18]:
tests.test_dataloaders(
    train_dataloader=yambda_train_dataloader,
    eval_dataloader=yambda_eval_dataloader,
)

All good! :)


In [19]:
train_batch = next(iter(yambda_train_dataloader))
eval_batch = next(iter(yambda_eval_dataloader))

print("TRAIN BATCH")
print({k: tuple(v.shape) for k, v in train_batch.items()})
print("uid[:5]:", train_batch["uid"][:5].tolist())
print("length[:5]:", train_batch["length"][:5].tolist())
print("history[:20]:", train_batch["history"][:20].tolist())
print("targets[:20]:", train_batch["targets"][:20].tolist())

print("\nEVAL BATCH")
print({k: tuple(v.shape) for k, v in eval_batch.items()})
print("uid[:5]:", eval_batch["uid"][:5].tolist())
print("length[:5]:", eval_batch["length"][:5].tolist())
print("history[:20]:", eval_batch["history"][:20].tolist())

TRAIN BATCH
{'uid': (128,), 'history': (8052,), 'length': (128,), 'targets': (8052,)}
uid[:5]: [300480, 504650, 161140, 375180, 727070]
length[:5]: [100, 89, 100, 61, 1]
history[:20]: [23996, 88763, 153033, 97903, 83733, 76621, 148890, 157109, 48307, 48747, 153444, 27115, 15610, 48307, 16735, 107765, 5297, 112963, 138689, 121784]
targets[:20]: [88763, 153033, 97903, 83733, 76621, 148890, 157109, 48307, 48747, 153444, 27115, 15610, 48307, 16735, 107765, 5297, 112963, 138689, 121784, 79562]

EVAL BATCH
{'uid': (128,), 'history': (8977,), 'length': (128,)}
uid[:5]: [30, 40, 50, 110, 130]
length[:5]: [100, 100, 100, 100, 20]
history[:20]: [147486, 4202, 77438, 99560, 4003, 78088, 79340, 31305, 78088, 78088, 146136, 117335, 140951, 124725, 58356, 49416, 35862, 53696, 45651, 8961]


# 2. Implement the Computation Graph: Transformer and Training Model That Returns the Loss

In this notebook, we will use a **GPT-style Transformer** as the sequence aggregation function. Unlike bidirectional `self-attention`, it uses a causal mask: the representation of an object at position `t` may depend only on the objects at positions `0, 1, ..., t`, but not on future elements of the sequence.

This approach is well suited for sequential modeling tasks, where the model must predict the next item using only the user's previous history.

In this part of the homework, you need to implement the main components of such a Transformer:

- causal Multi-Head Self-Attention;
- a position-wise MLP;
- a Transformer block;
- a stack of several Transformer blocks;
- the final model, which applies the Transformer to obtain user representations and computes the loss.

## Implementing the Attention Layer with a Causal Mask

Implement the `CausalSelfAttention` class, which performs `multi-head self-attention` with a causal mask.

The module receives:

- a tensor `x` of shape `(B, S, D)`, where:
  - `B` is the batch size (`batch_size`);
  - `S` is the sequence length (`seq_len`);
  - `D` is the hidden representation size (`embedding_dim`);
- a boolean mask `mask` of shape `(B, S)`, where `True` denotes a valid position and `False` denotes padding.

#### Requirements

You need to:

- project the input into `Q`, `K`, and `V` tensors;
- split them into `n_heads` attention heads;
- construct an attention mask that:
  - prevents attention to future positions;
  - prevents attention to padding positions;
- apply `scaled_dot_product_attention` with the causal mask;
- merge the attention heads back together;
- apply the final linear projection.

#### Why This Is Needed

This is the core mechanism of the Transformer model. It allows each element in the sequence to aggregate information from the preceding context. The causal mask guarantees that at position `t`, the model can use only elements from positions `<= t`, while the padding mask prevents artificial padded tokens from affecting the result.

### Data Flow Before `CausalSelfAttention`

```text
user histories
    ↓
Dataset
    ↓
collate_fn
    ↓
flattened histories + lengths
    ↓
create_masked_tensor
    ↓
padded item IDs + padding mask
    ↓
item embeddings
    ↓
x: [batch_size, seq_len, d_model]
    ↓
CausalSelfAttention
```


In [20]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        batch_size, seq_len, _ = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(
            batch_size, seq_len, self.n_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size, seq_len, self.n_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size, seq_len, self.n_heads, self.head_dim
        ).transpose(1, 2)

        causal_mask = torch.ones(
            seq_len,
            seq_len,
            dtype=torch.bool,
            device=x.device,
        ).tril()

        padding_mask = mask[:, None, None, :]
        attention_mask = causal_mask[None, None, :, :] & padding_mask # allowed = causal AND valid key

        attention_output = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )

        attention_output = attention_output.transpose(1, 2).contiguous() # transposing back
        attention_output = attention_output.view(
            batch_size,
            seq_len,
            self.d_model,
        )

        return self.out_proj(attention_output)

In [21]:
tests.test_causal_self_attention(CausalSelfAttention)

All good! :)


## MLP Implementation

Implement a two-layer `position-wise MLP` that is applied independently to every position in the sequence.

The module receives a tensor `x` of shape `(B, S, D)` and returns a tensor of the same shape, where:

- `B` is the batch size;
- `S` is the sequence length;
- `D` is the hidden representation size.

A typical MLP structure is:

1. linear projection `d_model -> 4 * d_model`;
2. `GELU` nonlinearity;
3. `dropout`;
4. linear projection `4 * d_model -> d_model`.

#### Why This Is Needed

The `self-attention` module mixes information across different positions in the sequence, while the `MLP` performs a nonlinear transformation independently within each position.

Together, these two components form a standard `Transformer block`.

In [22]:
class MLP(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.0):
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

        self.model = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        return self.model(x)

In [23]:
tests.test_mlp(MLP)

All good! :)


Essentially, `attention` determines where to obtain information, while the `MLP` determines what nonlinear computation to perform with the obtained information

## Transformer Block Implementation

Implement a single `Transformer block` using the **Pre-LayerNorm** formulation.

The `Transformer block` must consist of two consecutive residual blocks:

- `LayerNorm -> CausalSelfAttention -> Dropout -> Residual`
- `LayerNorm -> MLP -> Dropout -> Residual`

In other words, the input `x` must first be processed by the `CausalSelfAttention` block and then by the `MLP` block.

#### Why This Is Needed

A Transformer is built by applying several identical blocks sequentially. Each block gradually improves the representations of the sequence elements by combining:

- contextual interaction through the attention mechanism;
- nonlinear feature processing through the `MLP`.

In [24]:
class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

        self.ln_attn = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.attn_dropout = nn.Dropout(dropout)

        self.ln_mlp = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)
        self.mlp_dropout = nn.Dropout(dropout)


    def forward(
        self, x: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        if mask is None:
          mask = torch.ones(
              x.shape[:2],
              dtype=torch.bool,
              device=x.device,
          )

        x = x + self.attn_dropout(
            self.attn(self.ln_attn(x), mask)
        )

        x = x + self.mlp_dropout(
            self.mlp(self.ln_mlp(x))
        )

        return x

In [25]:
tests.test_block(Block)

All good! :)


## Encoder Implementation

Implement a **GPT-style encoder** that receives already constructed sequence representations and processes them sequentially through a stack of Transformer blocks.

#### The Model Must

1. Accept an input tensor `x` of shape `(B, S, D)`.
2. Apply `dropout` to the input representations.
3. Pass the result sequentially through `n_layers` Transformer blocks.
4. Apply a final `LayerNorm`.

The model must return hidden states of shape `(B, S, D)`.

#### Why This Is Needed

This is the main sequential model that builds contextual representations for every position in the sequence while respecting the causal constraint.

These representations can later be used to predict the next item in the sequence.

In [26]:
class GPT(nn.Module):
    def __init__(
        self,
        max_seq_len: int,
        n_layers: int,
        d_model: int,
        n_heads: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

        self.max_seq_len = max_seq_len

        self.input_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList(
            [
                Block(
                    d_model=d_model,
                    n_heads=n_heads,
                    dropout=dropout,
                )
                for _ in range(n_layers)
            ]
        )

        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        x = self.input_dropout(x)

        for block in self.blocks:
            x = block(x, mask)

        x = self.final_norm(x)

        return x

In [27]:
tests.test_gpt(GPT)

All good! :)


## UserEncoder

Implement the `UserEncoder` class, which builds contextual user representations at every point in the user's interaction history using a **GPT-style Transformer**.

The module receives a batch of user histories in flattened format.

#### What the Model Should Do

1. Convert `history` from item IDs into embeddings using a lookup layer (`nn.Embedding`).
2. Use `length` to reconstruct the boundaries of individual user sequences and convert the flattened representation into a padded representation.
3. Add positional embeddings.
4. Pass the resulting sequences through the `GPT` encoder.
5. Return the contextual representations of all users in flattened format.

If the total number of events in the batch is `N` and the representation dimension is `D`, then the output must have shape `(N, D)`, matching the flattened input structure.

#### Why This Is Needed

`UserEncoder` transforms interaction histories into a set of contextual representations, where each hidden state corresponds to a particular sequence position and incorporates information from all preceding events.

These representations can then be used to train the model on the `next item prediction` task.

In [28]:
class UserEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        assert embedding_dim % n_heads == 0, (
            "embedding_dim must be divisible by n_heads"
        )

        self.max_seq_len = max_seq_len

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim,
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            embedding_dim,
        )

        self.gpt = GPT(
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            d_model=embedding_dim,
            n_heads=n_heads,
            dropout=dropout,
        )


    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        history = inputs["history"]
        lengths = inputs["length"]

        item_embeddings = self.item_embedding(history) # items

        padded_embeddings, mask = create_masked_tensor( # rectangular format for transformer
            item_embeddings,
            lengths,
        )

        seq_len = padded_embeddings.shape[1]

        positions = torch.arange( # positions
            seq_len,
            device=history.device,
        )

        positional_embeddings = self.position_embedding(positions)

        x = padded_embeddings + positional_embeddings.unsqueeze(0)
        x = x.masked_fill(~mask.unsqueeze(-1), 0.0) # zeroing of padded positions

        hidden_states = self.gpt(x, mask)

        flat_hidden_states = hidden_states[mask] # flatten

        return flat_hidden_states

Reasoning behind zeroing out padded positions: padding creates empty slots, positional embeddings accidentally fill those empty slots with nonzero vectors, and zeroing restores them to truly empty positions before the GPT encoder

In [29]:
tests.test_user_encoder(UserEncoder)

All good! :)


## TrainNIPModel

Implement the `TrainNIPModel` class, which combines `UserEncoder` with the training logic for the `next-item prediction` task.

#### General Idea

In this assignment, we will build a complete two-tower model.

The user tower is `UserEncoder`. It receives a sequence of events from the user's history and builds a hidden representation for every prefix of that history.

The item tower is a simple item embedding table (`nn.Embedding`). For each item, its representation is obtained through a regular lookup in this matrix.

It is important that the same embedding matrix is used in both towers:

- as the representation of input events passed into `UserEncoder`;
- as the representation of positive target items and negative items when computing the loss.

In other words, there is no separate complex item encoder here. The item tower consists of a single embedding layer.

#### Model Input

The model receives a batch `inputs` containing:

- `history` — item sequences from user histories;
- `targets` — shifted targets used for training.

`history` is passed into `UserEncoder`, which returns hidden states for all positions of all histories in the batch.

`targets` is used to compute the loss.

#### The `forward` Function

The function must:

1. Pass `inputs` through `UserEncoder`.
2. Obtain the hidden states for all history positions.
3. Pass these hidden states together with `inputs` into `compute_loss`.
4. Return the resulting `loss`.

#### How Training Works

We want to train the model on the `next-item prediction` task.

Suppose a user's history is:

$$(i_1, i_2, \cdots, i_L).$$

Then the hidden state at position $t$ should be used to predict the next item, namely $i_{t+1}$.

However, in this assignment, the training dataset already constructs pairs of the following form:

- `history` = $[i_1, i_2, \dots, i_{L-1}]$;
- `targets` = $[i_2, i_3, \dots, i_L]$.

Therefore, the model is trained autoregressively over all sequence positions at once:

- the representation for $i_1$ is used to predict $i_2$;
- the representation for $i_2$ is used to predict $i_3$;
- ...
- the representation for $i_{L-1}$ is used to predict $i_L$.

Thus, `encoder_output` contains representations of user-history prefixes, while `targets` already contains the positive item for every position.

#### The `compute_loss` Function

In `compute_loss`, implement sampled softmax with in-batch negatives and the original, non-fixed, `logq` correction, as in the second homework assignment.

The idea is:

- `encoder_output` is used as the user representation for every sequence position;
- the positive item for every position is taken from `targets`;
- negative items are formed from other target items in the current batch;
- `logq` correction is applied to the logits.

#### LogQ Correction

When using in-batch negatives, different items appear in the batch with different frequencies. As a result, the negative-sampling distribution is not uniform.

To correct this bias, apply `logq` correction in the same way as in the second homework assignment.

The `compute_loss` function must:

1. Take the hidden representations from `encoder_output` and the corresponding labels from `targets`.
2. Obtain embeddings for these target items through the item embedding matrix.
3. Construct a logits matrix between the user representations and the in-batch candidate items.
4. Apply `logq` correction to the candidate logits.
5. Compute the final `loss`.

As a result, the model learns to build a representation for every prefix of the user's history so that it is close to the representation of the next interacted item and separated from the representations of other catalog items.

### Preparing Statistics for LogQ Correction

`TrainNIPModel` uses `logq` correction for in-batch negatives. In this part of the homework, you need to prepare **raw counts** over the training targets rather than precomputed frequencies.

Why:

- we do not want to store and pass a separate frequency vector `q`;
- for numerical stability, it is more convenient to work in log space;
- `logq` can be computed directly inside `forward` or `compute_loss` from the counts:

```text
logq(i) = log(count(i)) - log(sum_j count(j))
```

What Needs to Be Implemented:

- collect all target items from the training dataset;
- count how many times each `item_id` appears among the training targets;
- return a tensor of length `catalog_size`, where `q_counts[i] = count(item_i)`.

Important:

- compute the statistics over `targets`, not over `history`;
- do not normalize the counts into probabilities here;
- normalization and logarithms are performed inside the loss during `forward`.

In [30]:
def build_q_from_train_targets(
    train_targets: torch.Tensor,
    catalog_size: int,
) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    train_targets = train_targets.reshape(-1).long()

    if train_targets.numel() == 0:
        raise ValueError("train_targets must be non-empty")

    if catalog_size <= 0:
        raise ValueError("catalog_size must be positive")

    if train_targets.min() < 0 or train_targets.max() >= catalog_size:
        raise ValueError("train_targets contain invalid item ids")

    counts = torch.bincount(
        train_targets,
        minlength=catalog_size,
    ).float()

    return counts[:catalog_size]



train_target_ids = torch.tensor(
    [
        target
        for idx in range(len(yambda_train_dataset))
        for target in yambda_train_dataset[idx]["targets"]
    ],
    dtype=torch.long,
)


q = build_q_from_train_targets(
    train_targets=train_target_ids,
    catalog_size=catalog_size,
)

In [31]:
tests.test_logq_coefficients(build_q_from_train_targets)

All good! :)


In [32]:
class TrainNIPModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim,
        )

        self.encoder = UserEncoder(
            num_items=num_items,
            embedding_dim=embedding_dim,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            n_heads=n_heads,
            dropout=dropout,
        )

        # force sharing: UserEncoder and TrainNIPModel use the same table
        self.encoder.item_embedding = self.item_embedding

        self.num_negatives = num_negatives
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer("q_counts", q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if "weight" in key:
                nn.init.trunc_normal_(
                    value.data,
                    std=initializer_range,
                    a=-2 * initializer_range,
                    b=2 * initializer_range,
                )
            elif "bias" in key:
                nn.init.zeros_(value.data)

    def compute_loss(
        self, encoder_output: torch.Tensor, inputs: Dict[str, Any]
    ) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

        target_ids = inputs["targets"].long().reshape(-1)
        device = target_ids.device

        if target_ids.numel() == 0:
            return encoder_output.sum() * 0.0

        n = target_ids.shape[0]

        assert encoder_output.shape[0] == n, (
            f"encoder_output has {encoder_output.shape[0]} rows, "
            f"but target_ids has {n} rows"
        )

        if hasattr(self.encoder, "item_embeddings"):
            item_embedding = self.encoder.item_embeddings
        else:
            item_embedding = self.encoder.item_embedding

        # random in-batch negatives sampled by positions in target_ids.
        negative_pos = torch.randint(
            0,
            n,
            (n, self.num_negatives),
            device=device,
        )

        negative_ids = target_ids[negative_pos]  # [N, K]

        # candidate 0 is the positive item.
        candidate_ids = torch.cat(
            [
                target_ids[:, None],
                negative_ids,
            ],
            dim=1,
        )

        candidate_embeddings = item_embedding(candidate_ids)

        # dot product + logq correction
        logits = (encoder_output[:, None, :] * candidate_embeddings).sum(dim=-1)

        q_counts = self.q_counts.to(device=device)
        total_count = q_counts.sum().clamp_min(self.eps)

        neg_counts = q_counts[negative_ids].clamp_min(self.eps)
        neg_logq = torch.log(neg_counts) - torch.log(total_count)

        logits[:, 1:] = logits[:, 1:] - neg_logq

        labels = torch.zeros(n, dtype=torch.long, device=device)

        loss = F.cross_entropy(logits, labels)

        return loss


    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        loss = self.compute_loss(encoder_output, inputs)
        return loss

In [33]:
tests.test_train_nip_model(TrainNIPModel)

All good! :)


# 3. Implement model training

Implement the `train` function, which trains the model.

This time, we will not run validation during training yet. Therefore, the training loop includes only:

- iterating over all batches of the train dataset;
- computing the loss;
- performing optimization steps.

The function must return **two separate variables**:

- `checkpoint` — the model state after training, i.e. `state_dict`;
- `epoch_losses` — a list of average train losses by epoch, used to monitor training quality.

Expected return format:

```python
return checkpoint, epoch_losses

In [34]:
def train(
    dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    device: str = "cuda",
) -> tuple[dict[str, Any], list[float]]:
    model = model.to(device)

    epoch_losses = []

    for epoch in range(num_epochs):
        model.train()

        batch_losses = []

        for batch in dataloader: # making sure everything's on the same device
            batch = {
                key: value.to(device) if torch.is_tensor(value) else value
                for key, value in batch.items()
            }

            optimizer.zero_grad()

            loss = model(batch) # NIPModel - it'll call .forward()

            loss.backward()

            optimizer.step()

            batch_losses.append(loss.item())

        mean_epoch_loss = sum(batch_losses) / len(batch_losses)
        epoch_losses.append(mean_epoch_loss)

    checkpoint = model.state_dict()

    return checkpoint, epoch_losses

In [35]:
gc.collect()
torch.cuda.empty_cache()

train_graph = TrainNIPModel(
    num_items=catalog_size,
    embedding_dim=64,
    num_negatives=512,
    q_counts=q,
).to(DEVICE)
optimizer = torch.optim.Adam(params=train_graph.parameters(), lr=LEARNING_RATE)

_, epoch_losses = train(
    dataloader=yambda_train_dataloader,
    model=train_graph,
    optimizer=optimizer,
    num_epochs=1,
)

tests.test_train_loop(epoch_losses[0])

All good! :)


# 4. Implement model validation

## EvalNIPModel

Implement the `EvalNIPModel` class, which is used during inference and model evaluation.

#### General Idea

During training, the model built a representation for every prefix of the user's history in order to predict the next item from that prefix.

During inference, we usually do not need representations for all positions at once. Instead, we need one final user representation based on the last available event in the user's history.

#### Model Input

The model receives a batch `inputs` containing `history`.

`history` describes sequences of user events. After passing it through `UserEncoder`, we obtain hidden states for all positions of all histories in the batch.

#### The `forward` Function

The function must:

- pass `inputs` into `UserEncoder`;
- obtain hidden states for all history positions;
- for each user, select the hidden state corresponding to the last event in their history;
- compute relevance scores between each user and each item in the catalog;
- return a `scores` matrix of shape `[B, num_items]`, where:
  - `B` is the number of users in the batch;
  - `num_items` is the number of items in the catalog.

#### How Scores Are Computed

Relevance is computed as the dot product between the final user representation and the embedding of each item in the catalog.

For each user, we obtain a score for every item.


In [ ]:
class EvalNIPModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

In [ ]:
tests.test_eval_nip_model(EvalNIPModel)

## Function `eval`

Implement the `eval` function, which iterates over the whole dataset and evaluates the quality of the recommendation model.

#### General Idea

During evaluation, the model must, for each user:

1. Build relevance scores over the whole catalog.
2. Select the top-k most relevant items.
3. Store these recommendations.
4. After processing the whole dataset, compute the final quality metrics.

#### What the Function Does

The function must:

- switch the model to evaluation mode using `model.eval()`;
- iterate over all batches from `dataloader`;
- for each batch, obtain a score matrix of shape `[B, num_items]`;
- for each user, select the top-k items with the highest scores;
- collect all predictions into a dictionary of the following format: `Dict[uid, List[item_id]]`;
- after processing the whole dataset, call the provided `evaluate_fn(...)` function with the ground-truth `targets` and return the metrics dictionary.

#### Tips & Tricks

- Do not forget to switch the model to evaluation mode with `model.eval()`.
- Gradients are not needed during evaluation, so use `torch.no_grad()` or `torch.inference_mode()`.
- After obtaining catalog scores, select the best top-k items using `torch.topk`.
- Metrics are computed and averaged over all users in the dataset, not over a single batch.
- The function signature accepts `targets` and `evaluate_fn`, for example your `evaluate` function from the previous part of the homework, so the implementation does not depend on global variables.


In [ ]:
def eval(
    dataloader: DataLoader,
    model: torch.nn.Module,
    catalog_size: int,
    topk: int,
    device: str = "cuda",
    *,
    targets: Dict[int, List[int]],
    evaluate_fn: Callable[..., Dict[str, float]],
) -> Dict[str, float]:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
tests.test_eval(eval)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

train_graph = TrainNIPModel(
    num_items=catalog_size,
    embedding_dim=64,
    num_negatives=512,
    q_counts=q,
).to(DEVICE)
optimizer = torch.optim.Adam(params=train_graph.parameters(), lr=LEARNING_RATE)

checkpoint, epoch_losses = train(
    dataloader=yambda_train_dataloader,
    model=train_graph,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
)

test_dataset = yambda_eval_dataset
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False,
)

test_graph = EvalNIPModel(
    num_items=catalog_size,
    embedding_dim=64,
).to(DEVICE)
encoder_state = {
    key.removeprefix("encoder."): value
    for key, value in checkpoint.items()
    if key.startswith("encoder.")
}
test_graph.encoder.load_state_dict(encoder_state)

final_metrics_nip = eval(
    dataloader=test_dataloader,
    model=test_graph,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE,
    targets=targets,
    evaluate_fn=evaluate,
)
print(final_metrics_nip)
tests.check_nip_recs(final_metrics_nip)